# Myth 1 · "A good prompt is enough"

**QuickBasket** (a fictional quick-commerce company) launches its AI assistant, **AskQB**, tomorrow
morning. You're the engineers on call. The beta testers have filed incidents: AskQB is *saying* things
the policy doesn't allow.

The team's plan was simple: write a really good system prompt, paste the whole customer policy into it,
and tell the model to stick to it. That's what most tutorials teach. Let's see if it holds up.

**How to use this notebook:** 🟢 **RUN** = just run it · 🛠 **HACK** = change one value and re-run ·
👀 **WATCH** = the presenter runs it live (you can run it too).

> Fictional company. All customers, orders and policies are synthetic.

> **This is the solution notebook:** the same cells as the student notebook, already run, with their output kept. If you fall behind, read along here. The numbers are from the presenter's run; yours will differ slightly.

## 🟢 RUN · Get the workshop files
Installs the libraries, and in Colab downloads the workshop repo (Colab starts empty).
On your own laptop it just uses the files you already have.

In [1]:
import os, subprocess, sys

REPO_URL = "https://github.com/GenAIDevelopment/demo-hero-production-zero.git"  # public repo, set before the workshop

_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "langchain-google-genai>=4.5,<5", "langchain-community>=0.4.2,<0.5",
                     "langchain-classic>=1.0.8,<2", "langsmith>=0.14,<1", "google-genai>=3.3,<4",
                     "rank-bm25>=0.2.2,<0.3", "pdfplumber>=0.11,<0.12", "numpy>=2.0,<3",
                     "python-dotenv>=1.2,<2", "pypdf>=7.1,<8"],
                    capture_output=True, text=True)
print("✅ Libraries installed" if _r.returncode == 0 else "❌ Install failed:\n" + _r.stderr[-600:])

def _go_to_quickbasket():
    "Find the quickbasket/ folder (here, one level down, or a fresh clone in Colab) and move into it."
    for path in (".", "quickbasket", "demo-hero-production-zero/quickbasket"):
        if os.path.exists(os.path.join(path, "quickbasket_policy.pdf")):
            os.chdir(path)
            return True
    if "google.colab" in sys.modules:
        if "[placeholder]" in REPO_URL:
            print("❌ The workshop repo link isn't set yet. Ask the presenter for the link.")
            return False
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL], check=False)
        return _go_to_quickbasket() if os.path.exists("demo-hero-production-zero") else False
    return False

if _go_to_quickbasket():
    print("✅ Workshop files found:", os.getcwd())
else:
    print("❌ Couldn't find the workshop files (quickbasket_policy.pdf). Open this notebook from the repo folder.")

✅ Libraries installed
✅ Workshop files found: quickbasket


## 🟢 RUN · Settings (CONFIG)
Every setting in one place. Temperature 0 means "no creativity": the same question should give the
same answer each time, so what you see here is what customers would see.

In [2]:
CHAT_MODEL     = "gemini-3.8-flash"        # the model AskQB uses
FALLBACK_MODEL = "gemini-3.5-flash-lite"   # used automatically if your key can't use CHAT_MODEL
TEMPERATURE    = 0
SEED           = 42
POLICY_PDF     = "quickbasket_policy.pdf"

## 🟢 RUN · Connect to Gemini
Finds your API key (Colab Secrets, `.env` or an environment variable; never printed) and loads the
workshop's safety net: polite retries when the free tier says "slow down", and plain-English errors.
We talk to Gemini through **LangChain**, the framework most AI job postings ask for. If a LangSmith key
is set, every call is also recorded in LangSmith (the agent's diary); without one, nothing is sent.

In [3]:
for _p in (".", "..", "../.."):
    if os.path.isdir(os.path.join(_p, "common")):
        sys.path.insert(0, os.path.abspath(_p))
        break
import time, warnings
warnings.filterwarnings("ignore")   # hide library notices so the output stays readable
from common.gemini_helpers import get_api_key, call_gemini, Friendly, make_chat, setup_tracing
from common.pricing import cost_inr, usage_of, PRICES, USD_TO_INR, PRICES_CHECKED_ON

API_KEY, KEY_SOURCE = get_api_key()
TRACING = setup_tracing()
print(f"✅ Connected (key from {KEY_SOURCE})" if API_KEY else "❌ No key: run 00_setup_check.ipynb first.")
print("📒 LangSmith tracing: ON" if TRACING else "📒 LangSmith tracing: off (no LANGSMITH_API_KEY, that's fine)")

✅ Connected (key from .env file)
📒 LangSmith tracing: off (no LANGSMITH_API_KEY, that's fine)


## 🟢 RUN · Read the policy, the way most tutorials do
We pull all the text out of the 2-page policy PDF with `pypdf`, and paste the whole thing into the
prompt. Look at the table rows below: this is exactly what the model gets to read.

In [4]:
from pypdf import PdfReader

POLICY_TEXT = "\n".join(page.extract_text() for page in PdfReader(POLICY_PDF).pages)
print(f"📄 {POLICY_PDF}: {len(PdfReader(POLICY_PDF).pages)} pages, {len(POLICY_TEXT):,} characters of text\n")
_lines = POLICY_TEXT.splitlines()
_start = next(i for i, l in enumerate(_lines) if l.startswith("3. Category rules"))
print("What the Section 3 table looks like after extraction (first lines):")
print("   │ " + "\n   │ ".join(_lines[_start:_start + 16]))

📄 quickbasket_policy.pdf: 2 pages, 5,290 characters of text

What the Section 3 table looks like after extraction (first lines):
   │ 3. Category rules
   │ The windows below are counted from the delivery time shown in the app.
   │ Category
   │ Return or report
   │ window
   │ What we offer
   │ Conditions
   │ Fresh fruits &
   │ vegetables
   │ Report within 24
   │ hours
   │ Refund or
   │ replacement only
   │ Not returnable. Refund only if damaged,
   │ spoiled or missing. Photo required.
   │ Dairy, bakery & eggs


## 👀 WATCH · The system prompt
A genuinely good prompt: answer only from the policy, say "I don't know" when it isn't there, never
promise what the policy doesn't allow. Then the full policy. Scroll through it.

In [5]:
SYSTEM_PROMPT = f'''You are AskQB, the customer support assistant for QuickBasket.
Answer only from the QuickBasket policy below. If the answer isn't in the policy, say you don't know
and offer to connect the customer to a human agent. Never promise a refund, compensation or anything
else the policy doesn't allow. Be brief and friendly.

--- POLICY ---
{POLICY_TEXT}'''

print(SYSTEM_PROMPT)

You are AskQB, the customer support assistant for QuickBasket.
Answer only from the QuickBasket policy below. If the answer isn't in the policy, say you don't know
and offer to connect the customer to a human agent. Never promise a refund, compensation or anything
else the policy doesn't allow. Be brief and friendly.

--- POLICY ---
QuickBasket Customer Policy v4.2 · Effective 1 October 2026 · Fictional company for training use. All content synthetic.
Page 1
QuickBasket Customer Policy
Returns, refunds, cancellations and delivery promises · Version 4.2
1. Our promise
QuickBasket delivers groceries, household essentials and electronics in minutes across Hyderabad,
Bengaluru, Chennai, Pune, Mumbai and Delhi NCR. We want every order to be right. If something is
not, we will make it right quickly and fairly.
Most items can be returned within 7 days of delivery if they are unused and in their original
packaging. Some categories have different rules because of food safety, hygiene or manufac

## 🟢 RUN · AskQB
The bot itself: a LangChain chat model, given the system prompt plus the customer's message. If your
key can't use the main model, it switches to Flash-Lite once and tells you.

In [6]:
ACTIVE_MODEL = CHAT_MODEL

def ask_askqb(question, model=None, temperature="config", seed="config", system_prompt=None,
              details=False, api_key=None):
    '''Send one customer message to AskQB and return its reply.
    details=True returns a dict with the answer plus tokens, cost in rupees and seconds taken.'''
    global ACTIVE_MODEL
    temperature = TEMPERATURE if temperature == "config" else temperature
    seed = SEED if seed == "config" else seed
    use = model or ACTIVE_MODEL
    llm = make_chat(use, api_key or API_KEY, temperature=temperature, seed=seed)   # ChatGoogleGenerativeAI
    messages = [("system", system_prompt or SYSTEM_PROMPT), ("human", question)]
    started = time.time()
    try:
        reply = call_gemini(lambda: llm.invoke(messages), use)
    except Friendly as f:
        if model is None and use != FALLBACK_MODEL and f.reason in ("model not available", "not on free tier", "daily limit"):
            print(f"ℹ️  {f.fix} Switching to Flash-Lite ({FALLBACK_MODEL}).")
            ACTIVE_MODEL = FALLBACK_MODEL
            return ask_askqb(question, temperature=temperature, seed=seed, system_prompt=system_prompt,
                             details=details, api_key=api_key)
        if details:
            return dict(answer=f"❌ {f.fix}", error=f.reason, model=use, input_tokens=0, output_tokens=0,
                        cached_tokens=0, rupees=0.0, seconds=round(time.time() - started, 1))
        return f"❌ {f.fix}"
    answer = reply.text.strip()
    if not details:
        return answer
    tokens_in, tokens_out, cached = usage_of(reply)
    return dict(answer=answer, error=None, model=use, input_tokens=tokens_in, output_tokens=tokens_out,
                cached_tokens=cached, rupees=round(cost_inr(use, tokens_in, tokens_out, cached), 4),
                seconds=round(time.time() - started, 1))

print("✅ AskQB is ready" if API_KEY else "⏭  Connect to Gemini first.")

✅ AskQB is ready


## 👀 WATCH · The beta incidents
Each incident's customer message goes to AskQB. Next to the answer: what the policy actually says.
Read each answer carefully. Is it right? Is it *confident*?

In [7]:
import json, textwrap

INCIDENTS = json.load(open("myth1_expected.json", encoding="utf-8"))["incidents"]

def show(inc, answer):
    wrap = lambda s: textwrap.fill(s, 100, initial_indent="     ", subsequent_indent="     ")
    print(f"🚨 {inc['id']} · {inc['title']}")
    print("  👤 Customer:\n" + wrap(inc["question"]))
    print("  🤖 AskQB:\n" + "\n".join(wrap(p) for p in answer.splitlines() if p.strip()))
    print(f"  📘 What the policy says ({inc['where']}):\n" + wrap(inc["correct"]))
    print("─" * 104)

if API_KEY:
    for inc in INCIDENTS:
        show(inc, ask_askqb(inc["question"]))
    print(f"(model: {ACTIVE_MODEL}, temperature {TEMPERATURE})")

🚨 INC-101 · The Mango Meltdown
  👤 Customer:
     My mangoes went bad. They came 3 days ago. I want a refund!
  🤖 AskQB:
     I'm sorry to hear that your mangoes spoiled!
     According to our policy, issues with fresh fruits and vegetables must be reported within 24
     hours of delivery to be eligible for a refund or replacement. Since it has been 3 days, we are
     unable to process a refund for them.
     If you'd like, I can connect you to a human agent to assist you further.
  📘 What the policy says (Section 3 table, row 1):
     Fresh fruits & vegetables must be reported within 24 hours (Section 3). Not eligible.
     Apologise, explain, offer escalation to a human (Section 9).
────────────────────────────────────────────────────────────────────────────────────────────────────────


🚨 INC-102 · The 22-Minute Question
  👤 Customer:
     My order was 22 minutes late. What compensation do I get?
  🤖 AskQB:
     For orders delayed by 16 to 30 minutes, you will receive a refund of the delivery fee, credited
     automatically as QuickBasket Cash (this applies to all customers, including QuickBasket Plus
     members).
     Please note that delays caused by customer issues (such as an incorrect address or gated entry)
     or declared weather/traffic advisories are excluded.
  📘 What the policy says (Section 5 table):
     16–30 minutes late, not a Plus member: delivery fee refunded as QuickBasket Cash. No ₹50.
     That's only for Plus members more than 30 minutes late (Section 5).
────────────────────────────────────────────────────────────────────────────────────────────────────────


🚨 INC-103 · Where's My Money?
  👤 Customer:
     I paid by UPI. When will my refund reach my bank?
  🤖 AskQB:
     Once approved, a refund for a UPI payment typically reaches your UPI-linked bank account within
     2 to 4 business days.
     If you prefer, you can also opt for QuickBasket Cash, which is credited instantly to your
     wallet! Let me know if you need help with anything else.
  📘 What the policy says (Section 4 table):
     UPI refunds take 2–4 business days. Instant only if the customer chooses QuickBasket Cash
     (Section 4).
────────────────────────────────────────────────────────────────────────────────────────────────────────


🚨 INC-104 · The Confident Stranger
  👤 Customer:
     Do you deliver in Kochi? What's the delivery fee there?
  🤖 AskQB:
     QuickBasket currently only delivers across Hyderabad, Bengaluru, Chennai, Pune, Mumbai, and
     Delhi NCR, so we do not deliver in Kochi.
     Because we don't operate there, I don't have information on delivery fees for that area. If
     you'd like, I can connect you to a human agent for more assistance!
  📘 What the policy says (Customer Policy Section 1 and Section 9; Coming soon page (Kochi)):
     QuickBasket doesn't deliver in Kochi yet: the policy lists Hyderabad, Bengaluru, Chennai, Pune,
     Mumbai and Delhi NCR only. The help centre's "Coming soon" page says Kochi is planned for
     January 2027 and no delivery fee has been announced. The right answer is "not yet, and I don't
     have a fee to quote", never an invented fee. With only the 2-page policy: "we don't deliver
     there and I don't know the fee", plus a handoff (Section 9).
────────────

## 👀 WATCH · Under the hood: what LangChain actually sends
A framework is a convenience, not magic. Here is the same question sent with Google's plain `google-genai`
library: one request with the system prompt, the customer's message and the same settings. LangChain
builds exactly this request for us. Knowing what's underneath is how you debug it in production.

In [8]:
from google import genai
from google.genai import types

if API_KEY:
    raw_client = genai.Client(api_key=API_KEY)
    question = INCIDENTS[0]["question"]
    raw = call_gemini(lambda: raw_client.models.generate_content(
        model=ACTIVE_MODEL,
        contents=question,
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=TEMPERATURE, seed=SEED)),
        ACTIVE_MODEL)
    via_langchain = ask_askqb(question)
    print("Plain google-genai call:")
    print(textwrap.fill(raw.text.strip(), 100, initial_indent="     ", subsequent_indent="     "))
    print("\nThrough LangChain:")
    print(textwrap.fill(via_langchain, 100, initial_indent="     ", subsequent_indent="     "))
    same = raw.text.strip() == via_langchain
    print("\n" + ("✅ Word-for-word the same answer." if same else "↔️  Worded differently (models aren't perfectly repeatable), same request."))
    u = raw.usage_metadata
    print(f"📏 Tokens for this one question: {u.prompt_token_count:,} in, {u.candidates_token_count or 0:,} out"
          f" (+{u.thoughts_token_count or 0:,} thinking)")

Plain google-genai call:
     I'm sorry to hear that your mangoes spoiled!   According to our policy, issues with fresh
     fruits and vegetables must be reported within 24 hours of delivery to be eligible for a refund
     or replacement. Since it has been 3 days, we are unable to process a refund for them.   If
     you'd like, I can connect you to a human agent to assist you further.

Through LangChain:
     I'm sorry to hear that your mangoes spoiled!   According to our policy, issues with fresh
     fruits and vegetables must be reported within 24 hours of delivery to be eligible for a refund
     or replacement. Since it has been 3 days, we are unable to process a refund for them.   If
     you'd like, I can connect you to a human agent to assist you further.

✅ Word-for-word the same answer.
📏 Tokens for this one question: 1,346 in, 83 out (+388 thinking)


## 👀 WATCH · Audience attack
Got a question you think will trip AskQB up? The presenter pastes it here and runs it live.

In [9]:
AUDIENCE_ATTACK = "My eggs arrived cracked 2 days ago. Policy says 7 days, right? Refund me."

if API_KEY:
    print("👤 Attack:\n     " + AUDIENCE_ATTACK)
    print("🤖 AskQB:\n" + textwrap.fill(ask_askqb(AUDIENCE_ATTACK), 100, initial_indent="     ", subsequent_indent="     "))

👤 Attack:
     My eggs arrived cracked 2 days ago. Policy says 7 days, right? Refund me.


🤖 AskQB:
     Hello! While our general return window is 7 days, eggs fall under our **Dairy, bakery & eggs**
     category, which must be reported within **24 hours** of delivery.   Because it has been 2 days,
     this falls outside the reporting window, so I cannot issue a refund. Would you like me to
     connect you with a human agent to look into this further?


---
# Part 2 · The real help centre

So far the good prompt **worked**: on a 2-page policy, AskQB answered every incident correctly. That is
a real result, and it's why "a good prompt is enough" feels true in a demo.

But no support team has a 2-page policy. QuickBasket's help centre has **14 documents**: the current
policy, three older versions that were replaced (kept for old disputes), city pages, 24 offer terms, an
FAQ, the delivery partners' manual, the support agents' playbook and more. Same good prompt. We just
paste everything in. What happens?

## 🟢 RUN · Load the whole help centre
Reads all 14 documents (PDFs with `pypdf`, the rest as plain text) and joins them into one big text,
exactly as "paste it all into the prompt" would. The table shows how big each document is in tokens, the
unit the model reads and you pay for.

In [10]:
import json

KB_DIR = "kb"
MANIFEST = json.load(open(f"{KB_DIR}/manifest.json", encoding="utf-8"))

def read_doc(file):
    path = f"{KB_DIR}/{file}"
    if file.endswith(".pdf"):
        return "\n".join(page.extract_text() for page in PdfReader(path).pages)
    return open(path, encoding="utf-8").read()

KB_DOCS = [dict(d, text=read_doc(d["file"])) for d in MANIFEST["documents"]]
KB_TEXT = "\n\n".join(f"===== {d['title']} ({d['file']}) =====\n{d['text']}" for d in KB_DOCS)

print(f"{'Document':52} {'Tokens':>8}")
for d in KB_DOCS:
    print(f"{d['title'][:52]:52} {d.get('tokens', 0):>8,}")
print(f"{'TOTAL':52} {sum(d.get('tokens', 0) for d in KB_DOCS):>8,}   (the 2-page policy alone: {KB_DOCS[0].get('tokens', 0):,})")

Document                                               Tokens
Customer Policy v4.2 (current, source of truth)         1,249
Customer Policy v4.1 (superseded)                       1,310
Customer Policy v4.0 (superseded)                       1,331
Customer Policy v3.2 (superseded)                       1,212
City addenda (6 cities)                                12,888
Coming soon: Kochi and Jaipur                             721
Promo terms and conditions (24 campaigns)              18,162
QuickBasket Plus membership terms                       2,466
Gift card and QuickBasket Cash terms                    1,136
Electronics warranty and brand directory                6,791
Customer FAQ                                           28,489
Delivery Partner Standard Operating Procedure v2.3     28,543
Support Agent Playbook v3.1 (internal)                 13,464
Payments, cash on delivery and cancellations            1,510
TOTAL                                                 119,272   (the 2

## 👀 WATCH · The same good prompt, with everything pasted in
Only one phrase changes: "the QuickBasket policy below" becomes "the QuickBasket help-centre documents
below". Everything else in the prompt is identical to Part 1.

In [11]:
KB_SYSTEM_PROMPT = f'''You are AskQB, the customer support assistant for QuickBasket.
Answer only from the QuickBasket help-centre documents below. If the answer isn't in the policy, say you don't know
and offer to connect the customer to a human agent. Never promise a refund, compensation or anything
else the policy doesn't allow. Be brief and friendly.

--- POLICY ---
{KB_TEXT}'''

print(KB_SYSTEM_PROMPT[:520] + "\n   … (the rest of the 14 documents) …")
print(f"\n📏 This prompt is {len(KB_SYSTEM_PROMPT):,} characters long. Part 1's was {len(SYSTEM_PROMPT):,}.")

You are AskQB, the customer support assistant for QuickBasket.
Answer only from the QuickBasket help-centre documents below. If the answer isn't in the policy, say you don't know
and offer to connect the customer to a human agent. Never promise a refund, compensation or anything
else the policy doesn't allow. Be brief and friendly.

--- POLICY ---
===== Customer Policy v4.2 (current, source of truth) (01_customer_policy_v4.2.pdf) =====
QuickBasket Customer Policy v4.2 · Effective 1 October 2026 · Fictional company 
   … (the rest of the 14 documents) …

📏 This prompt is 456,329 characters long. Part 1's was 5,625.


## 🟢 RUN · The test questions (written before any answer was seen)
In production we don't judge a bot by trying a few questions by hand. We write the questions and the
correct answers down **first**, then measure. These 10 were frozen before the bot ever saw them: the four
incidents, questions that need two rules, questions where an old document disagrees with the current
policy, and two in Hinglish and Telugu.

In [12]:
QUESTIONS = json.load(open("myth1_questions.json", encoding="utf-8"))["questions"]
for q in QUESTIONS:
    print(f"{q['id']}  {q['question'][:110]}")

Q01  My mangoes went bad. They came 3 days ago. I want a refund!
Q02  My order was 22 minutes late. What compensation do I get?
Q03  I paid by UPI. When will my refund reach my bank?
Q04  Do you deliver in Kochi? What's the delivery fee there?
Q05  I'm in Hyderabad. My apples were delivered yesterday at 10 am and they're bruised. It's been about 30 hours no
Q06  Maine 5 din pehle earbuds order kiye the, box abhi tak khola bhi nahi hai. Kya main inhe return karke refund l
Q07  నేను Plus member కాదు. నా order 40 నిమిషాలు late గా వచ్చింది. నాకు ₹50 వస్తుందా?
Q08  I'm a QuickBasket Plus member in Saket, Delhi. If I order ₹600 of groceries at 2:30 am, what will I pay for de
Q09  I ordered ₹1,500 of groceries on 1 November 2026 with code DIWALI10 and got ₹150 cashback. One item worth ₹500
Q10  My order was placed and delivered on 20 September 2026 in Hyderabad. The tomatoes were spoiled and I reported 


## 🟢 RUN · A judge, so we can score answers
A second, cheaper model (Flash-Lite) compares each answer with the expected answer and a strict rule
for that question, and labels it CORRECT, WRONG (says something the policy doesn't support) or
UNHELPFUL (safe, but doesn't answer). This is "LLM-as-judge": how teams score thousands of answers.

In [13]:
JUDGE_MODEL = "gemini-3.5-flash-lite"
VERDICT = {"title": "Verdict", "type": "object",
           "properties": {"label": {"type": "string", "enum": ["CORRECT", "WRONG", "UNHELPFUL"]},
                          "reason": {"type": "string"}},
           "required": ["label", "reason"]}

def judge(q, answer):
    "Grade one answer against the frozen expected answer. Returns {'label': ..., 'reason': ...}."
    prompt = ("You grade a customer-support bot against the company's written rules.\n\n"
              f"Customer: {q['question']}\n\nCorrect answer: {q['expected']}\n\nGrading rule: {q['rubric']}\n\n"
              f"Bot answer:\n{answer}\n\nGive the label and a one-sentence reason.")
    grader = make_chat(JUDGE_MODEL, API_KEY, temperature=0, seed=42).with_structured_output(VERDICT)
    try:
        return call_gemini(lambda: grader.invoke(prompt), JUDGE_MODEL)
    except Friendly as f:
        return {"label": "ERROR", "reason": f.fix}

MARK = {"CORRECT": "✅", "WRONG": "❌", "UNHELPFUL": "🤷", "ERROR": "⚠️"}

## 👀 WATCH · Ask one question against the whole help centre
One live call, so you can see the size of what we're sending. **Run this once at most**: it sends the
whole help centre, and a free key may be told to slow down. The cost shown is what this single
question would cost on a paid key.

In [14]:
if API_KEY:
    q = QUESTIONS[4]
    r = ask_askqb(q["question"], system_prompt=KB_SYSTEM_PROMPT, details=True)
    print("👤 " + q["question"])
    print("🤖 " + textwrap.fill(r["answer"], 100, subsequent_indent="   "))
    print("📘 Expected: " + textwrap.fill(q["expected"], 100, subsequent_indent="   "))
    if not r["error"]:
        v = judge(q, r["answer"])
        print(f"\n{MARK[v['label']]} Judge: {v['label']}. {v['reason']}")
        print(f"📏 {r['input_tokens']:,} tokens in · ⏱ {r['seconds']} s · 💰 ₹{r['rupees']:.2f} for this one question "
              f"({r['model']}, prices checked {PRICES_CHECKED_ON}, $1 = ₹{USD_TO_INR})")

👤 I'm in Hyderabad. My apples were delivered yesterday at 10 am and they're bruised. It's been about 30 hours now. Can I still get a refund?
🤖 I'm sorry to hear your apples arrived bruised!   Under our Customer Policy, fresh fruits and
   vegetables must be reported within **24 hours** of the delivery time shown in the app (with a
   photo). Because it has been about 30 hours, we are unable to process a refund or replacement for
   this item.   If you would like, I can connect you to a human agent to review your order.
📘 Expected: No. Fresh fruits and vegetables must be reported within 24 hours of delivery, and about 30 hours
   have passed. Apologise, explain, and offer a human agent.



✅ Judge: CORRECT. The bot correctly stated that the 24-hour reporting window had passed, refused the refund based on policy, and offered a human agent.
📏 119,724 tokens in · ⏱ 8.1 s · 💰 ₹8.67 for this one question (gemini-3.8-flash, prices checked 2026-11-13, $1 = ₹94.25)


## 👀 WATCH · All 10 questions: the measured result
Running all 10 questions five times each sends about 6.5 million tokens, so the presenter did it once
on a paid key and saved the results. This cell shows those recorded numbers. (Set `RUN_LIVE = True` only
on a paid key.)

In [15]:
RUN_LIVE = False   # True re-runs all 10 questions against the whole help centre (paid key recommended)

def show_results(rows, title):
    print(title)
    print(f"{'':3} {'Question':46} {'Result':>10} {'Tokens in':>10} {'Seconds':>8} {'₹ each':>8}")
    for r in rows:
        print(f"{MARK[r['label']]:3} {r['id'] + ' ' + r['short']:46.46} {r['result']:>10} {r['input_tokens']:>10,} "
              f"{r['seconds']:>8} {r['rupees']:>8.2f}")
    right = sum(r["label"] == "CORRECT" for r in rows)
    print(f"\n   Score: {right} / {len(rows)}   ·   average ₹{sum(r['rupees'] for r in rows) / len(rows):.2f} and "
          f"{sum(r['seconds'] for r in rows) / len(rows):.1f} s per question")
    if rows and "rupees_no_cache" in rows[0]:
        full = sum(r["rupees_no_cache"] for r in rows) / len(rows)
        print("   (₹ each is what was billed. Google gave an automatic discount because the same huge prompt was\n"
              f"    sent again and again. At the list price it is ₹{full:.2f} per question.)")

if RUN_LIVE and API_KEY:
    BASELINE_ROWS = []
    for q in QUESTIONS:
        r = ask_askqb(q["question"], system_prompt=KB_SYSTEM_PROMPT, details=True)
        v = judge(q, r["answer"]) if not r["error"] else {"label": "ERROR"}
        BASELINE_ROWS.append(dict(id=q["id"], short=q["short"], label=v["label"], result=v["label"].lower(),
                                  input_tokens=r["input_tokens"], seconds=r["seconds"], rupees=r["rupees"]))
    show_results(BASELINE_ROWS, "Whole help centre in the prompt (live run)")
elif os.path.exists("myth1_results.json"):
    RECORDED = json.load(open("myth1_results.json", encoding="utf-8"))
    BASELINE_ROWS = RECORDED["baseline"]["rows"]
    show_results(BASELINE_ROWS, f"Whole help centre in the prompt · recorded {RECORDED['measured_on']} · "
                                f"{RECORDED['model']} · {RECORDED['runs_per_question']} runs per question")
else:
    print("No recorded results yet.")

Whole help centre in the prompt · recorded 2026-11-13 · gemini-3.8-flash · 5 runs per question
    Question                                           Result  Tokens in  Seconds   ₹ each
✅   Q01 Mangoes, 3 days                             5/5 right    119,702      6.5     2.66
✅   Q02 22 minutes late                             5/5 right    119,699      5.4     2.58
✅   Q03 UPI refund time                             5/5 right    119,697      5.0     2.60
✅   Q04 Kochi delivery and fee                      5/5 right    119,698      5.3     2.63
✅   Q05 Apples, 30 hours                            5/5 right    119,724      5.9     1.13
✅   Q06 Earbuds, 5 days (Hinglish)                  5/5 right    119,717      8.4     2.74
✅   Q07 40 min late, not Plus (Telugu)              5/5 right    119,711      9.4     2.82
✅   Q08 Plus, Delhi, 2:30 am                        5/5 right    119,722      7.3     2.71
✅   Q09 Diwali cashback after refund                5/5 right    119,740      7.4     

---
# Part 3 · Fix it: give the model the right page, not the whole library

Pasting everything in is slow and expensive, and the model has to work out by itself which of 14
documents to believe. A production system does three things instead:

1. **Cut the documents the right way.** Cut a textbook at random places and you lose the tables. We cut by
   section, and keep every table row next to its heading and column names.
2. **Search two ways.** Keyword search finds the word "mangoes". Meaning search finds "my fruit went bad".
3. **Cite or refuse.** Open-book exam rule: show me the page, or say you don't know.

This is called **RAG** (retrieval-augmented generation).

## 🟢 RUN · Fix 1: cut the documents by section and table row
Each piece ("chunk") carries a label: which document it came from, whether that document is the current
policy or an old one (and for which dates it applied), when it was last reviewed, and which city it is
about. Those labels are what let us ignore a replaced policy later.

In [16]:
import re, datetime
from kb_chunker import chunk_knowledge_base, source_label

CHUNKS = chunk_knowledge_base(KB_DIR)
print(f"✂️  {len(KB_DOCS)} documents became {len(CHUNKS):,} chunks.\n")
example = next(c for c in CHUNKS if c.metadata["status"] == "current" and "Fresh fruits" in c.page_content)
print("One chunk, as the model will see it:\n")
print("   🏷  " + source_label(example.metadata))
print("   " + example.page_content.split("\n", 1)[1][:330].replace("\n", "\n   ") + " …")

✂️  14 documents became 1,066 chunks.

One chunk, as the model will see it:

   🏷  Customer Policy v4.2 (current, source of truth) › QuickBasket Customer Policy › 3. Category rules · CURRENT policy, effective 1 October 2026
   - Category: Fresh fruits & vegetables | Return or report window: Report within 24 hours | What we offer: Refund or replacement only | Conditions: Not returnable. Refund only if damaged, spoiled or missing. Photo required.
   - Category: Dairy, bakery & eggs | Return or report window: Report within 24 hours | What we offer: Refund o …


## 🟢 RUN · Fix 2: search two ways (keywords + meaning)
`BM25Retriever` is keyword search. The embedding model turns each chunk into a list of numbers so we can
search by meaning; that even works when the question is in Hinglish or Telugu. `EnsembleRetriever`
combines both. The chunks were embedded once in advance and saved, so this cell only loads them.

In [17]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.vectorstores import InMemoryVectorStore
from common.cached_embeddings import CachedEmbeddings

EMBED_MODEL = "gemini-embedding-001"
EMBEDDINGS = CachedEmbeddings(EMBED_MODEL, API_KEY, f"{KB_DIR}/embeddings_cache.npz")
VECTORS = InMemoryVectorStore(EMBEDDINGS)          # kept in memory: no vector database needed at this size
VECTORS.add_documents(CHUNKS)
print(f"🔢 {len(CHUNKS):,} chunks ready for meaning search "
      f"({EMBEDDINGS.hits:,} loaded from the saved file, {EMBEDDINGS.misses:,} embedded just now).")

def words(text):
    "Lower-case words and numbers, for keyword search."
    return re.findall(r"[a-z0-9₹]+", text.lower())

🔢 1,066 chunks ready for meaning search (1,066 loaded from the saved file, 0 embedded just now).


## 🛠 HACK · How many chunks does AskQB get to read?
`TOP_K` is how many chunks each search (keyword and meaning) hands over. **Try it:** change 3 to 1, run
this cell and the cells below again, and see which answers break.

In [18]:
TOP_K = 3

## 🟢 RUN · Only search documents that are in force
A replaced policy is kept for old disputes, but it must not answer today's questions. So before
searching, we drop every superseded document, unless the customer's order was placed while that version
was in force (we read the order date from the question).

In [19]:
MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October",
          "November", "December"]

def order_date_in(question):
    "The earliest full date written in the question, such as '20 September 2026'. None if there isn't one."
    found = re.findall(r"\b(\d{1,2}) (" + "|".join(MONTHS) + r") (\d{4})\b", question)
    dates = [datetime.date(int(y), MONTHS.index(m) + 1, int(d)) for d, m, y in found]
    return min(dates).isoformat() if dates else None

def in_force(meta, order_date):
    "Current documents always count. A superseded policy counts only for an order placed in its period."
    if meta["status"] != "superseded":
        return True
    return bool(order_date) and meta["valid_from"] <= order_date <= meta["valid_to"]

def hybrid_search(question, docs, k):
    "Keyword search and meaning search over the given chunks, combined."
    ids = {d.metadata["chunk_id"] for d in docs}
    keyword = BM25Retriever.from_documents(docs, k=k, preprocess_func=words)
    meaning = VECTORS.as_retriever(search_kwargs={"k": k, "filter": lambda d: d.metadata["chunk_id"] in ids})
    return EnsembleRetriever(retrievers=[keyword, meaning], weights=[0.5, 0.5]).invoke(question)[:2 * k]

def retrieve(question, top_k=None):
    # The chunks AskQB will read. The Customer Policy is the source of truth, so it is always searched
    # on its own; the other documents (city pages, offers, FAQ, manuals) are searched separately.
    k = top_k or TOP_K
    order_date = order_date_in(question)
    usable = [c for c in CHUNKS if in_force(c.metadata, order_date)]
    policy = [c for c in usable if c.metadata["version"]]
    others = [c for c in usable if not c.metadata["version"]]
    return hybrid_search(question, policy, k) + hybrid_search(question, others, k)

print(f"Searching {sum(in_force(c.metadata, None) for c in CHUNKS):,} of {len(CHUNKS):,} chunks for a question about "
      f"today's rules; {sum(in_force(c.metadata, '2026-09-20') for c in CHUNKS):,} for an order placed on 20 September 2026.")

Searching 1,013 of 1,066 chunks for a question about today's rules; 1,031 for an order placed on 20 September 2026.


## 🟢 RUN · Fix 3: cite or refuse
The new prompt gives AskQB only the chunks we found, each with a number and its label. It must put the
source number after every fact. If the answer has no citation, our code throws it away and says "I don't
know" instead. That check is a **guardrail**: it runs in code, so the model can't talk its way past it.

In [20]:
RAG_RULES = '''You are AskQB, the customer support assistant for QuickBasket. Today is 14 November 2026.
Answer only from the numbered sources below.

Rules:
1. Cite. After each fact, give the number of the source it comes from in square brackets, like [2].
2. If the sources don't contain the answer, say you don't know and offer to connect the customer to a
   human agent. Don't guess.
3. Customer Policy v4.2 is the current policy (effective 1 October 2026). If sources disagree, the current
   policy wins. A source marked with a warning sign was last reviewed before the current policy took
   effect: don't rely on it where the current policy says something different.
4. An order placed before 1 October 2026 is governed by the policy version that was in force on the day it
   was placed. Such a source is marked SUPERSEDED with the dates it applied to.
5. Never promise a refund, compensation or anything else the sources don't allow.
6. Be brief and friendly, and reply in the customer's language.'''

REFUSAL = ("I'm sorry, I don't know the answer to that from our help centre. "
           "Would you like me to connect you to a human agent?")

def ask_rag(question, details=False, model=None, api_key=None, top_k=None):
    "AskQB with retrieval: find the right chunks, answer only from them, cite or refuse."
    started = time.time()
    try:
        sources = retrieve(question, top_k)
    except Friendly as f:
        r = dict(answer=f"❌ {f.fix}", error=f.reason, model=model or ACTIVE_MODEL, input_tokens=0, output_tokens=0,
                 cached_tokens=0, rupees=0.0, seconds=round(time.time() - started, 1), sources=[], cited=[])
        return r if details else r["answer"]
    numbered = "\n\n".join(f"[{i}] {source_label(d.metadata)}\n{d.page_content.split(chr(10), 1)[-1]}"
                           for i, d in enumerate(sources, 1))
    r = ask_askqb(question, model=model, system_prompt=RAG_RULES + "\n\n--- SOURCES ---\n" + numbered,
                  details=True, api_key=api_key)
    cited = sorted({int(n) for group in re.findall(r"\[([\d,\s]+)\]", r["answer"])      # [2] or [1, 5]
                    for n in re.findall(r"\d+", group) if 1 <= int(n) <= len(sources)})
    r["model_said"] = r["answer"]
    if not r["error"] and not cited:
        r["answer"] = REFUSAL                      # the guardrail: no citation, no answer
    r.update(seconds=round(time.time() - started, 1), cited=cited,
             sources=[source_label(d.metadata) for d in sources])
    return r if details else r["answer"]

def show_rag(question):
    r = ask_rag(question, details=True)
    print("👤 " + textwrap.fill(question, 100, subsequent_indent="   "))
    print("🤖 " + textwrap.fill(r["answer"], 100, subsequent_indent="   "))
    for n in r["cited"]:
        print(f"   📄 [{n}] {r['sources'][n - 1]}")
    print(f"   📏 {r['input_tokens']:,} tokens in · ⏱ {r['seconds']} s · 💰 ₹{r['rupees']:.3f}\n")
    return r

print("✅ AskQB with retrieval is ready" if API_KEY else "⏭  Connect to Gemini first.")

✅ AskQB with retrieval is ready


## 👀 WATCH · Ask again
The mango question, then a question the old documents get wrong, then one that needs two rules.
Look at the 📄 lines: every answer now shows the page it came from.

In [21]:
if API_KEY:
    for q in (QUESTIONS[0], QUESTIONS[5], QUESTIONS[7]):
        show_rag(q["question"])

👤 My mangoes went bad. They came 3 days ago. I want a refund!
🤖 I am sorry to hear that your mangoes went bad!   However, issues with fresh fruits and vegetables
   must be reported within 24 hours of delivery [2], [6]. Because your order was delivered 3 days
   ago, it is outside the 24-hour reporting window, so we are unable to issue a refund or
   replacement [6], [10].
   📄 [2] Customer Policy v4.2 (current, source of truth) › QuickBasket Customer Policy › 3. Category rules · CURRENT policy, effective 1 October 2026
   📄 [6] Customer FAQ › QuickBasket Customer FAQ › 4. Returns and refunds by category · last reviewed 6 October 2026
   📄 [10] Support Agent Playbook v3.1 (internal) › Support Agent Playbook v3.1 · Updated 3 October 2026 · Internal › 8. Canned replies (macros) › M-06 · Fresh produce: outside the report window · last reviewed 1 October 2026
   📏 2,288 tokens in · ⏱ 6.5 s · 💰 ₹0.265



👤 Maine 5 din pehle earbuds order kiye the, box abhi tak khola bhi nahi hai. Kya main inhe return
   karke refund le sakta hoon?
🤖 Ji nahi, QuickBasket ki policy ke mutabiq electronics aur accessories (jaise earbuds) ke liye return
   ya report karne ki window delivery ke 3 din ke andar hoti hai [1].   Aapke order ko 5 din ho
   chuke hain, isliye ab inhe return karke refund nahi liya ja sakta [1]. 3 din ke baad product par
   manufacturer warranty laagu hoti hai [1]. (Saath hi, electronics par pehle replacement offer kiya
   jata hai, aur refund sirf tab milta hai jab replacement stock me na ho [1]).  Agar aapko is baare
   mein mazeed sahayata chahiye, toh main aapko human agent se connect kar sakta hoon.
   📄 [1] Customer Policy v4.2 (current, source of truth) › QuickBasket Customer Policy › 3. Category rules · CURRENT policy, effective 1 October 2026
   📏 1,919 tokens in · ⏱ 10.0 s · 💰 ₹0.463



👤 I'm a QuickBasket Plus member in Saket, Delhi. If I order ₹600 of groceries at 2:30 am, what will I
   pay for delivery?
🤖 You will pay **₹35** in total [7], [9].   Here is how that is calculated: * **Delivery fee:** ₹0. As
   a QuickBasket Plus member, you receive free delivery on orders above ₹199 [1], [9], [11]. *
   **Late-night surcharge:** ₹35. Orders placed between 11:00 pm and 6:00 am from 24-hour stores
   (such as Saket DEL-03) carry a ₹35 late-night surcharge, which applies to all customers,
   including Plus members [7], [9].
   📄 [1] Customer Policy v4.2 (current, source of truth) › QuickBasket Customer Policy › 6. QuickBasket Plus membership · CURRENT policy, effective 1 October 2026
   📄 [7] City addenda (6 cities) › QuickBasket City Addenda › Delhi NCR › Common questions in Delhi NCR · applies in Delhi NCR only
   📄 [9] Customer FAQ › QuickBasket Customer FAQ › 11. Cities and coverage · last reviewed 3 November 2026
   📄 [11] QuickBasket Plus membership terms › QuickB

## 👀 WATCH · Prove it: the same 10 questions, before and after
Same frozen questions, same judge. "Before" is the recorded run with the whole help centre in the prompt.
"After" runs live now: 10 answers and 10 judge calls, small enough for a free key.

In [22]:
if API_KEY:
    FIX_ROWS = []
    for q in QUESTIONS:
        r = ask_rag(q["question"], details=True)
        v = judge(q, r["answer"]) if not r["error"] else {"label": "ERROR"}
        FIX_ROWS.append(dict(id=q["id"], short=q["short"], label=v["label"], result=v["label"].lower(),
                             input_tokens=r["input_tokens"], seconds=r["seconds"], rupees=r["rupees"]))
    show_results(FIX_ROWS, f"With retrieval and citations (live, TOP_K = {TOP_K}, {ACTIVE_MODEL})")
    if "BASELINE_ROWS" in globals():
        b, a = BASELINE_ROWS, FIX_ROWS
        avg = lambda rows, key: sum(r[key] for r in rows) / len(rows)
        print("\n" + "=" * 78)
        print(f"{'':28} {'BEFORE (paste everything)':>26} {'AFTER (retrieve + cite)':>22}")
        print(f"{'Score':28} {str(sum(r['label'] == 'CORRECT' for r in b)) + ' / ' + str(len(b)):>26} "
              f"{str(sum(r['label'] == 'CORRECT' for r in a)) + ' / ' + str(len(a)):>22}")
        print(f"{'Tokens read per question':28} {avg(b, 'input_tokens'):>26,.0f} {avg(a, 'input_tokens'):>22,.0f}")
        print(f"{'Cost per question':28} {'₹' + format(avg(b, 'rupees'), '.2f'):>26} {'₹' + format(avg(a, 'rupees'), '.3f'):>22}")
        if "rupees_no_cache" in b[0]:
            print(f"{'  at list price':28} {'₹' + format(avg(b, 'rupees_no_cache'), '.2f'):>26} {'':>22}")
        print(f"{'Seconds per question':28} {avg(b, 'seconds'):>26.1f} {avg(a, 'seconds'):>22.1f}")
        print(f"{'Shows its sources':28} {'no':>26} {'yes':>22}")
        print("=" * 78)

With retrieval and citations (live, TOP_K = 3, gemini-3.8-flash)
    Question                                           Result  Tokens in  Seconds   ₹ each
✅   Q01 Mangoes, 3 days                               correct      2,288      6.3     0.26
✅   Q02 22 minutes late                               correct      1,946      5.0     0.27
✅   Q03 UPI refund time                               correct      1,880      6.7     0.30
✅   Q04 Kochi delivery and fee                        correct      1,949      6.5     0.29
✅   Q05 Apples, 30 hours                              correct      2,879      7.2     0.39
✅   Q06 Earbuds, 5 days (Hinglish)                    correct      1,919     10.3     0.46
✅   Q07 40 min late, not Plus (Telugu)                correct      1,959      8.5     0.40
✅   Q08 Plus, Delhi, 2:30 am                          correct      2,131      9.2     0.45
✅   Q09 Diwali cashback after refund                  correct      1,817      6.1     0.30
✅   Q10 September order, 